In [1]:
import pandas as pd
import numpy as np

# Load raw dataset
DATA_PATH = "../data/raw/Online_Retail.xlsx"

df = pd.read_excel(DATA_PATH)

# Keep a copy of the original data
df_raw = df.copy()

print("Original dataset:", df.shape)

Original dataset: (541909, 8)


In [2]:
# Count exact duplicates
duplicate_count = df.duplicated().sum()

print("Number of duplicates:", duplicate_count)

# Display duplicate examples
duplicates = df[df.duplicated(keep=False)]

duplicates.head(10)

Number of duplicates: 5268


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [3]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (536641, 8)


In [4]:
# Convert identifiers to strings
df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()

df["StockCode"] = df["StockCode"].astype("string").str.strip()

# Convert customer IDs while preserving missing values
df["CustomerID"] = pd.to_numeric(
    df["CustomerID"],
    errors="coerce"
).astype("Int64")

# Convert date
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

# Convert numerical columns
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Clean product descriptions
df["Description"] = df["Description"].astype("string").str.strip()

df.info()

<class 'pandas.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  string        
 1   StockCode    536641 non-null  string        
 2   Description  535187 non-null  string        
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[us]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  Int64         
 7   Country      536641 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(1), string(3)
memory usage: 37.4 MB


In [5]:
# Identify cancelled invoices
df["IsCancellation"] = df["InvoiceNo"].str.startswith(
    "C",
    na=False
)

# Identify invalid transactions
df["InvalidQuantity"] = df["Quantity"].isna() | (df["Quantity"] <= 0)

df["InvalidPrice"] = df["UnitPrice"].isna() | (df["UnitPrice"] <= 0)

df["InvalidDate"] = df["InvoiceDate"].isna()

In [6]:
quality_report = pd.DataFrame({
    "Issue": [
        "Missing CustomerID",
        "Missing Description",
        "Cancelled invoices",
        "Invalid quantities",
        "Invalid prices",
        "Invalid dates"
    ],
    "Count": [
        df["CustomerID"].isna().sum(),
        df["Description"].isna().sum(),
        df["IsCancellation"].sum(),
        df["InvalidQuantity"].sum(),
        df["InvalidPrice"].sum(),
        df["InvalidDate"].sum()
    ]
})

quality_report

,Issue,Count
0,Missing CustomerID,135037
1,Missing Description,1454
2,Cancelled invoices,9251
3,Invalid quantities,10587
4,Invalid prices,2512
5,Invalid dates,0


In [7]:
valid_sales_mask = (
    ~df["IsCancellation"]
    & ~df["InvalidQuantity"]
    & ~df["InvalidPrice"]
    & ~df["InvalidDate"]
)

sales_clean = df.loc[valid_sales_mask].copy()

# Create revenue column
sales_clean["Revenue"] = (
    sales_clean["Quantity"] * sales_clean["UnitPrice"]
)

print("Clean sales dataset:", sales_clean.shape)

print(
    "Total revenue:",
    round(sales_clean["Revenue"].sum(), 2)
)

Clean sales dataset: (524878, 13)
Total revenue: 10642110.8


In [8]:
customer_sales = sales_clean[
    sales_clean["CustomerID"].notna()
].copy()

print("Customer sales dataset:", customer_sales.shape)

print(
    "Unique customers:",
    customer_sales["CustomerID"].nunique()
)

Customer sales dataset: (392692, 13)
Unique customers: 4338


In [9]:
from pathlib import Path

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned sales
sales_clean.to_csv(
    output_dir / "sales_clean.csv",
    index=False
)

# Save customer sales
customer_sales.to_csv(
    output_dir / "customer_sales.csv",
    index=False
)

# Save quality report
quality_report.to_csv(
    output_dir / "data_quality_report.csv",
    index=False
)

print("Files successfully exported!")

Files successfully exported!


In [10]:
# Validation checks

assert sales_clean["Quantity"].gt(0).all()

assert sales_clean["UnitPrice"].gt(0).all()

assert sales_clean["InvoiceDate"].notna().all()

assert not sales_clean["IsCancellation"].any()

assert sales_clean["Revenue"].notna().all()

assert customer_sales["CustomerID"].notna().all()

print("All validation checks passed!")

All validation checks passed!


In [11]:
print("Sales clean:", sales_clean.shape)
print("Customer sales:", customer_sales.shape)

print("Missing CustomerID in sales_clean:",
      sales_clean["CustomerID"].isna().sum())

print("Unique customers:",
      customer_sales["CustomerID"].nunique())

print("Total revenue:",
      round(sales_clean["Revenue"].sum(), 2))

Sales clean: (524878, 13)
Customer sales: (392692, 13)
Missing CustomerID in sales_clean: 132186
Unique customers: 4338
Total revenue: 10642110.8
